In [1]:
# 1. Vérification de la carte graphique NVIDIA allouée par Google
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# 2. Installation de la bibliothèque CuPy (version optimisée pour CUDA 12)
!pip install -q cupy-cuda12x

print("\n✅ Installation terminée et GPU prêt !")

Tesla T4, 15360 MiB

✅ Installation terminée et GPU prêt !


In [2]:
# ==========================================
# CELLULE 1 : BENCHMARK ISOLÉ ET VALIDATION (EXERCICE 25)
# ==========================================
!pip install -q gradio
!wget -q -nc https://files.grouplens.org/datasets/movielens/ml-1m.zip
!unzip -q -n ml-1m.zip

import time
import numpy as np
import pandas as pd
import cupy as cp

print("📥 Chargement du dataset MovieLens 1M...")
ratings = pd.read_csv('ml-1m/ratings.dat', sep='::', engine='python', names=['user_id', 'movie_id', 'rating', 'timestamp'])
pivot_df = ratings.pivot(index='user_id', columns='movie_id', values='rating').fillna(0)
matrice_complete = pivot_df.values.astype(np.float32)

# Normalisation globale
normes = np.linalg.norm(matrice_complete, axis=1, keepdims=True)
matrice_complete = matrice_complete / (normes + 1e-8)

print("\n📊 RUNNING ACADEMIC BENCHMARK (Moyenne sur 3 essais avec perf_counter)...")
print("=" * 80)
print(f"{'Taille Matrice':<20} | {'Temps Moyen CPU':<18} | {'Temps Moyen GPU':<18} | {'Speedup':<10} | {'Validation'}")
print("=" * 80)

tailles_test = [1000, 3000, 6040]
N_REPETITIONS = 3

for taille in tailles_test:
    # Slicing de la matrice pour le test d'échelle
    A_cpu = matrice_complete[:taille]
    A_gpu = cp.asarray(A_cpu)

    # Warm-up CUDA initial pour cette taille
    _ = cp.dot(A_gpu, A_gpu.T)
    cp.cuda.Stream.null.synchronize()

    # --- Mesure CPU ---
    temps_cpu_runs = []
    for _ in range(N_REPETITIONS):
        t_start = time.perf_counter()
        sim_cpu = np.dot(A_cpu, A_cpu.T)
        temps_cpu_runs.append(time.perf_counter() - t_start)
    t_moyen_cpu = np.mean(temps_cpu_runs)

    # --- Mesure GPU ---
    temps_gpu_runs = []
    for _ in range(N_REPETITIONS):
        t_start = time.perf_counter()
        sim_gpu = cp.dot(A_gpu, A_gpu.T)
        cp.cuda.Stream.null.synchronize()
        temps_gpu_runs.append(time.perf_counter() - t_start)
    t_moyen_gpu = np.mean(temps_gpu_runs)

    # --- Validation Numérique (L'argument ultime demandé par Claude) ---
    sim_gpu_cpu = cp.asnumpy(sim_gpu)
    calcul_valide = np.allclose(sim_cpu, sim_gpu_cpu, atol=1e-4)
    status_valid = "✅ CPU == GPU" if calcul_valide else "❌ Erreur Numérique"

    speedup = t_moyen_cpu / t_moyen_gpu
    dim_str = f"{taille}x{matrice_complete.shape[1]}"
    print(f"{dim_str:<20} | {t_moyen_cpu:.4f}s           | {t_moyen_gpu:.4f}s           | x{speedup:.1f}       | {status_valid}")

print("=" * 80)
print("📌 L'exercice technique 25 est validé avec succès. Tu peux lancer l'application ci-dessous.")

📥 Chargement du dataset MovieLens 1M...

📊 RUNNING ACADEMIC BENCHMARK (Moyenne sur 3 essais avec perf_counter)...
Taille Matrice       | Temps Moyen CPU    | Temps Moyen GPU    | Speedup    | Validation
1000x3706            | 0.0621s           | 0.0020s           | x31.2       | ✅ CPU == GPU
3000x3706            | 0.3649s           | 0.0160s           | x22.9       | ✅ CPU == GPU
6040x3706            | 1.5126s           | 0.0870s           | x17.4       | ✅ CPU == GPU
📌 L'exercice technique 25 est validé avec succès. Tu peux lancer l'application ci-dessous.


In [3]:
# ==========================================
# CELLULE 2 : INTERFACE DE DÉMO APPLICATIVE (GPU-Flix V3)
# ==========================================
import gradio as gr
import matplotlib.pyplot as plt

custom_css = """
body, .gradio-container {
    background: #0b0f19 !important;
    color: #f8fafc !important;
    font-family: 'Segoe UI', system-ui, -apple-system, sans-serif !important;
}
.glow-box {
    background: linear-gradient(135deg, rgba(30, 41, 59, 0.8) 0%, rgba(15, 23, 42, 0.9) 100%);
    border: 1px solid rgba(0, 242, 254, 0.3);
    border-radius: 12px;
    padding: 20px;
    box-shadow: 0 8px 32px 0 rgba(0, 0, 0, 0.5);
}
.history-card {
    background: rgba(30, 41, 59, 0.4);
    border-left: 3px solid #fbbf24;
    border-radius: 6px;
    padding: 8px 14px;
    margin-bottom: 6px;
    font-size: 0.9em;
    display: flex;
    justify-content: space-between;
}
.movie-card {
    background: rgba(30, 41, 59, 0.6);
    border-left: 4px solid #00f2fe;
    border-radius: 8px;
    padding: 14px 18px;
    margin-bottom: 12px;
    transition: transform 0.2s;
    box-shadow: 0 4px 12px rgba(0,0,0,0.2);
}
.movie-card:hover {
    transform: translateX(6px);
    border-left-color: #4facfe;
    background: rgba(30, 41, 59, 0.9);
}
.movie-title {
    font-size: 1.1em;
    font-weight: bold;
    color: #ffffff;
    margin-bottom: 6px;
}
.progress-bar-bg {
    background: #1e293b;
    border-radius: 6px;
    height: 8px;
    width: 100%;
    overflow: hidden;
    margin-top: 8px;
}
.progress-bar-fill {
    background: linear-gradient(90deg, #00f2fe 0%, #4facfe 100%);
    height: 100%;
    border-radius: 6px;
}
"""


# Préparation des structures globales issues de la cellule 1
matrice_cpu = matrice_complete
matrice_gpu = cp.asarray(matrice_cpu)

movie_id_to_title = dict(zip(pd.read_csv('ml-1m/movies.dat', sep='::', engine='python', encoding='latin-1', names=['movie_id', 'title', 'genres'])['movie_id'], pd.read_csv('ml-1m/movies.dat', sep='::', engine='python', encoding='latin-1', names=['movie_id', 'title', 'genres'])['title']))
TITRES_REELS = [movie_id_to_title.get(mid, f"Film #{mid}") for mid in pivot_df.columns.tolist()]

def executer_recommandation_hpc(user_choice):
    plt.close('all') # Correction fuite mémoire Matplotlib
    user_idx = int(user_choice.split("#")[1].split(" ")[0]) - 1
    N_REPETITIONS = 3

    # --- 1. Mesure CPU Pure (Isolée et répétée) ---
    temps_cpu_runs = []
    for _ in range(N_REPETITIONS):
        start_cpu = time.perf_counter()
        sim_cpu = np.dot(matrice_cpu, matrice_cpu.T)
        temps_cpu_runs.append(time.perf_counter() - start_cpu)
    temps_cpu = np.mean(temps_cpu_runs)

    # Post-traitement hors chronomètre principal
    np.fill_diagonal(sim_cpu, 0)
    scores_cpu = np.dot(sim_cpu[user_idx], matrice_cpu)
    scores_cpu[matrice_cpu[user_idx] > 0] = -np.inf

    # --- 2. Mesure GPU Pure (Isolée, synchronisée et répétée) ---
    temps_gpu_runs = []
    for _ in range(N_REPETITIONS):
        start_gpu = time.perf_counter()
        sim_gpu = cp.dot(matrice_gpu, matrice_gpu.T)
        cp.cuda.Stream.null.synchronize()
        temps_gpu_runs.append(time.perf_counter() - start_gpu)
    temps_gpu = np.mean(temps_gpu_runs)

    # Post-traitement GPU hors chronomètre principal
    cp.fill_diagonal(sim_gpu, 0)
    scores_gpu = cp.dot(sim_gpu[user_idx], matrice_gpu)
    scores_gpu[matrice_gpu[user_idx] > 0] = -cp.inf
    cp.cuda.Stream.null.synchronize()

    speedup = temps_cpu / temps_gpu

    # --- 3. Rendu HTML de l'historique et des prédictions ---
    notes_brutes = pivot_df.iloc[user_idx].values
    top_history_idx = np.argsort(notes_brutes)[::-1][:3]

    html_cards = f"""
    <div style="margin-bottom: 15px;">
        <span style="color: #94a3b8; font-size: 0.85em; text-transform: uppercase; letter-spacing: 1px;">📜 Base de calcul : Ce que le client #{user_idx+1} a adoré</span>
    </div>
    """
    for idx in top_history_idx:
        if notes_brutes[idx] > 0:
            html_cards += f"""
            <div class="history-card">
                <span style="color: #e2e8f0;">✔ {TITRES_REELS[idx]}</span>
                <strong style="color: #fbbf24;">★ {notes_brutes[idx]:.1f}/5.0</strong>
            </div>
            """

    html_cards += f"""
    <div style="margin: 20px 0 12px 0; border-top: 1px solid #334155; padding-top: 15px;">
        <span style="color: #00f2fe; font-size: 0.9em; font-weight: bold; letter-spacing: 0.5px;">🚀 NOUVELLES PRÉDICTIONS GPU (EXCLUANT SES FILMS DÉJÀ VUS) :</span>
    </div>
    """

    scores_final = cp.asnumpy(scores_gpu)
    top_indices = np.argsort(scores_final)[::-1][:3]
    scores_valides = scores_final[scores_final != -np.inf]
    max_score = np.max(scores_valides) if len(scores_valides) > 0 else 1.0
    scores_norm = (scores_final / (max_score + 1e-8)) * 99.4

    for rank, idx in enumerate(top_indices):
        score = scores_norm[idx]
        html_cards += f"""
        <div class="movie-card">
            <div style="display: flex; justify-content: space-between; align-items: center;">
                <span class="movie-title">#{rank+1} — {TITRES_REELS[idx]}</span>
                <span style="color: #00f2fe; font-weight: bold; font-size: 0.95em;">{score:.1f}% Match</span>
            </div>
            <div class="progress-bar-bg"><div class="progress-bar-fill" style="width: {score}%;"></div></div>
        </div>
        """

    # --- 4. Rendu Graphique Matplotlib ---
    fig, ax = plt.subplots(figsize=(7, 3.8))
    fig.patch.set_facecolor('#0f172a')
    ax.set_facecolor('#1e293b')

    barres = ax.bar(['CPU (NumPy)', 'GPU (CuPy)'], [temps_cpu, temps_gpu], color=['#ef4444', '#00f2fe'], width=0.45)
    ax.set_ylabel('Temps MatMul pur (secondes)', color='#94a3b8', fontsize=10)
    ax.set_title(f'ACCÉLÉRATION MATRICIELLE PURE : x{speedup:.1f}', color='#ffffff', fontweight='bold', fontsize=13, pad=12)
    ax.tick_params(colors='#e2e8f0', labelsize=10)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#475569'); ax.spines['bottom'].set_color('#475569')
    ax.grid(axis='y', linestyle='--', alpha=0.2, color='#94a3b8')

    for barre in barres:
        yval = barre.get_height()
        ax.text(barre.get_x() + barre.get_width()/2.0, yval + (max(temps_cpu, temps_gpu)*0.03), f'{yval:.4f} s', ha='center', va='bottom', color='#ffffff', fontweight='bold', fontsize=10)

    plt.tight_layout()

    # Label de dimension corrigé selon la remarque de Claude (6040x6040 en sortie)
    status_box = f"⚡ BENCHMARK PUR : Multiplication [6040x3706] x [3706x6040] -> Résultat [{matrice_cpu.shape[0]}x{matrice_cpu.shape[0]}] calculé en {temps_gpu:.4f}s sur GPU vs {temps_cpu:.4f}s sur CPU (Moyenne sur {N_REPETITIONS} itérations)."
    return status_box, html_cards, fig

# --- Frontend Gradio ---
liste_users = [f"Utilisateur MovieLens #{i+1} (Profil Réel)" for i in [99, 0, 41, 500, 1023, 2048, 5000]]

with gr.Blocks() as app_web:
    with gr.Row():
        gr.HTML("""
        <div style="border-bottom: 2px solid #00f2fe; padding-bottom: 15px; margin-bottom: 10px; width: 100%;">
            <h1 style="color: #ffffff; margin: 0; font-size: 2.2em;">🚀 GPU-Flix <span style="font-size: 0.5em; background: #00f2fe; color: #000; padding: 3px 8px; border-radius: 4px; vertical-align: middle;">AI STREAMING ENGINE</span></h1>
            <p style="color: #94a3b8; margin: 5px 0 0 0;">Système de recommandation par filtrage collaboratif (S = A · A^T) — Accélération matérielle NVIDIA T4</p>
        </div>
        """)
    with gr.Row():
        with gr.Column(scale=5, elem_classes="glow-box"):
            selecteur = gr.Dropdown(choices=liste_users, value=liste_users[0], label="🎯 Échantillon de profils test (parmi les 6 040 clients réels)")
            btn_run = gr.Button("⚡ LANCER LE BENCHMARK INSTANTANÉ (CPU vs GPU)", variant="primary")
            statut_perf = gr.Textbox(label="📊 Télémétrie & Speedup MatMul (Isolé)", interactive=False)
            graphique = gr.Plot()
        with gr.Column(scale=6, elem_classes="glow-box"):
            zone_cartes = gr.HTML("""
            <div style="text-align: center; padding: 60px 20px; color: #64748b;">
                <h3 style="margin-bottom: 10px;">En attente d'exécution...</h3>
                <p>Cliquez sur le bouton de benchmark pour exécuter la multiplication matricielle sur la carte graphique et générer les recommandations.</p>
            </div>
            """)

    btn_run.click(fn=executer_recommandation_hpc, inputs=[selecteur], outputs=[statut_perf, zone_cartes, graphique])

app_web.launch(share=True, css=custom_css)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b46aaa08033c156132.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
